In [18]:
import cv2
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from skimage import io, color
from skimage.filters import threshold_otsu
from pyspark.sql import SparkSession
import numba

# Initialize Spark Session with GraphFrames package
spark = SparkSession.builder \
    .appName("ImageGraph") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.0-s_2.12") \
    .getOrCreate()


In [21]:
from pyspark.sql.functions import col, abs as sql_abs
from graphframes import GraphFrame

import cv2
import numpy as np

# Assuming Spark session has been started as shown above
def create_graph_from_image(image):
    rows, cols = image.shape
    vertices = []
    edges = []

    for r in range(rows):
        for c in range(cols):
            vertex_id = r * cols + c
            vertices.append((vertex_id, r, c, int(image[r, c])))

            if r > 0:
                neighbor_id = (r-1) * cols + c
                weight = abs(int(image[r, c]) - int(image[r-1, c]))
                edges.append((vertex_id, neighbor_id, weight))
                
            if c > 0:
                neighbor_id = r * cols + (c-1)
                weight = abs(int(image[r, c]) - int(image[r, c-1]))
                edges.append((vertex_id, neighbor_id, weight))
                
            if r > 0 and c > 0:
                neighbor_id = (r-1) * cols + (c-1)
                weight = abs(int(image[r, c]) - int(image[r-1, c-1]))
                edges.append((vertex_id, neighbor_id, weight))
                
            if r > 0 and c < cols-1:
                neighbor_id = (r-1) * cols + (c+1)
                weight = abs(int(image[r, c]) - int(image[r-1, c+1]))
                edges.append((vertex_id, neighbor_id, weight))

    # Create DataFrames for vertices and edges
    vertices_df = spark.createDataFrame(vertices, ["id", "row", "col", "intensity"])
    edges_df = spark.createDataFrame(edges, ["src", "dst", "weight"])

    # Create GraphFrame
    graph = GraphFrame(vertices_df, edges_df)

    return graph

# Load and preprocess the image
image_path = '/home/matheus/github/natural_artificial_vision/imagens_teste/folhas.png'  # Update with your image path
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
image = cv2.resize(image, (100, 100))  # Resize for simplicity

# Create the graph
graph = create_graph_from_image(image)

# Example: Display vertices and edges
print("Vertices:")
graph.vertices.show()

print("Edges:")
graph.edges.show()

# Stop Spark session after processing
spark.stop()


TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1mUntyped global name 'spark':[0m [1m[1mCannot determine Numba type of <class 'pyspark.sql.session.SparkSession'>[0m
[1m
File "../../../../tmp/ipykernel_688/1421893827.py", line 40:[0m
[1m<source missing, REPL/exec in use?>[0m
[0m

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from graphframes import GraphFrame
import cv2
import numpy as np
from itertools import combinations

# Initialize Spark Session with GraphFrames package
spark = SparkSession.builder \
    .appName("ImageGraph") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.2-spark3.0-s_2.12") \
    .getOrCreate()

def create_fully_connected_graph_from_image(image):
    rows, cols = image.shape
    vertices = []
    edges = []

    # Create a vertex for each pixel
    for r in range(rows):
        for c in range(cols):
            vertex_id = r * cols + c
            vertices.append((vertex_id, r, c, int(image[r, c])))

    # Create a fully connected graph by connecting each pair of nodes
    for (r1, c1), (r2, c2) in combinations([(r, c) for r in range(rows) for c in range(cols)], 2):
        vertex_id_1 = r1 * cols + c1
        vertex_id_2 = r2 * cols + c2
        weight = abs(int(image[r1, c1]) - int(image[r2, c2]))
        edges.append((vertex_id_1, vertex_id_2, weight))
        edges.append((vertex_id_2, vertex_id_1, weight))  # Since it's an undirected graph

    # Convert vertices and edges to DataFrames
    vertices_df = spark.createDataFrame(vertices, ["id", "row", "col", "intensity"])
    edges_df = spark.createDataFrame(edges, ["src", "dst", "weight"])

    # Create GraphFrame
    graph = GraphFrame(vertices_df, edges_df)

    return graph

# Load and preprocess the image
image_path = '/home/matheus/github/natural_artificial_vision/imagens_teste/folhas.png'  # Update with your image path
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
image = cv2.resize(image, (50, 50))  # Resize to avoid memory issues with full connections

# Create the fully connected graph
graph1 = create_fully_connected_graph_from_image(image)

# Example: Display vertices and edges
print("Vertices:")
graph1.vertices.show()

print("Edges:")
graph1.edges.show()

# Stop Spark session after processing
spark.stop()


Vertices:


24/08/24 09:07:20 WARN TaskSetManager: Stage 1 contains a task of very large size (2276 KiB). The maximum recommended task size is 1000 KiB.


+---+---+---+---------+
| id|row|col|intensity|
+---+---+---+---------+
|  0|  0|  0|      255|
|  1|  0|  1|      255|
|  2|  0|  2|      255|
|  3|  0|  3|      255|
|  4|  0|  4|      255|
|  5|  0|  5|      255|
|  6|  0|  6|      255|
|  7|  0|  7|      255|
|  8|  0|  8|      255|
|  9|  0|  9|      255|
| 10|  0| 10|      255|
| 11|  0| 11|      255|
| 12|  0| 12|      255|
| 13|  0| 13|      255|
| 14|  0| 14|      255|
| 15|  0| 15|      255|
| 16|  0| 16|      255|
| 17|  0| 17|      255|
| 18|  0| 18|      255|
| 19|  0| 19|      255|
+---+---+---+---------+
only showing top 20 rows

Edges:
+---+---+------+
|src|dst|weight|
+---+---+------+
|  0|  1|     0|
|  1|  0|     0|
|  0|  2|     0|
|  2|  0|     0|
|  0|  3|     0|
|  3|  0|     0|
|  0|  4|     0|
|  4|  0|     0|
|  0|  5|     0|
|  5|  0|     0|
|  0|  6|     0|
|  6|  0|     0|
|  0|  7|     0|
|  7|  0|     0|
|  0|  8|     0|
|  8|  0|     0|
|  0|  9|     0|
|  9|  0|     0|
|  0| 10|     0|
| 10|  0|     0|


In [13]:
print(graph1)

GraphFrame(v:[id: bigint, row: bigint ... 2 more fields], e:[src: bigint, dst: bigint ... 1 more field])


In [10]:
def find_borders(G, threshold):
    edges = []
    for (u, v, d) in G.edges(data=True):
        if d['weight'] > threshold:
            edges.append((u, v))
    return edges

In [11]:
def draw_borders(image, edges):
    for (u, v) in edges:
        image[u] = 255
        image[v] = 255
    return image


In [14]:
edges = find_borders(graph1.edges,30)

border_folhas = np.zeros_like(image)
border_folhas = draw_borders(border_folhas, edges)

AttributeError: 'DataFrame' object has no attribute 'edges'